# Healthcare Accessibility Classification: Identifying Underserved ZIP Codes in Houston

## 1. Introduction
This Exploratory Data Analysis (EDA) serves as the foundation for identifying "healthcare deserts" within the Houston, Texas, metropolitan area. This project simulates a real-world use case for a Healthcare Real Estate Investment Trust (REIT) aiming to optimize facility placement by identifying areas where high demographic demand meets low healthcare supply.

### 1.1 Objective
The primary goal is to build an end-to-end data science pipeline that classifies Houston-area ZIP codes as "underserved" based on the relationship between healthcare facility density and key demographic demand indicators.

### 1.2 Business Problem
**Problem Statement:** Where are high-demand areas (specifically those with an aging population) underserved by current healthcare infrastructure? 

To address this, the project will:
* Quantify healthcare facility density per ZIP code.
* Analyze the relationship between demographic demand and facility supply.
* Provide actionable insights for potential new facility investment.

### 1.3 Research Question
Can demographic indicators—specifically median age, total population, and median household income—accurately predict whether a Houston-area ZIP code is "underserved" (defined as having a facility density below the 25th percentile)?

### 1.4 Hypothesis
Houston ZIP codes with a higher-than-average median age and lower-than-average household income are significantly more likely to be classified as healthcare deserts (underserved) compared to more affluent, younger areas.

---

## 2. Data Overview
The analysis joins two primary public datasets by 5-digit ZIP code:

* **Supply Data:** CMS Provider of Services (POS) File, containing geographic and type data for hospitals, clinics, and other certified facilities.
* **Demand Data:** U.S. Census Bureau ACS 5-Year Estimates for median age, total population, and median household income.

In [41]:
# import standard libraries
import os
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [42]:
BASE_DIR = os.path.dirname(os.getcwd()) # Gets the root project directory
data_folder = os.path.join(BASE_DIR, "data")

In [43]:
# Load datasets
census_df = pd.read_csv(os.path.join(data_folder, "raw_census_data.csv"))
healthcare_df = pd.read_csv(os.path.join(data_folder, "raw_pos.csv"))
print("Datasets loaded successfully!")
print(f"Census shape: {census_df.shape}")
print(f"POS shape: {healthcare_df.shape}")

C:\Users\willi\AppData\Local\Temp\ipykernel_28168\3362416116.py:3: DtypeWarning: Columns (8,17,18,19,100) have mixed types. Specify dtype option on import or set low_memory=False.
  healthcare_df = pd.read_csv(os.path.join(data_folder, "raw_pos.csv"))


Datasets loaded successfully!
Census shape: (1989, 11)
POS shape: (77283, 182)


In [44]:
print("Census Data Sample:")
print(census_df.head())
print("\nPoint of Service Data Sample:")
print(healthcare_df.head())

Census Data Sample:
   zip_code  median_age  total_population  median_household_income  \
0     75001        33.8           16633.0                  77598.0   
1     75002        38.8           72679.0                 119301.0   
2     75006        36.2           48062.0                  79672.0   
3     75007        40.6           54498.0                 105969.0   
4     75009        37.1           28109.0                 153658.0   

   bachelors_degree  masters_degree  below_poverty  unemployed  \
0            4875.0          1965.0         1404.0       400.0   
1           16165.0          6082.0         3890.0      1732.0   
2            7733.0          2357.0         3777.0      1057.0   
3           12001.0          3981.0         4141.0      1215.0   
4            6690.0          1960.0         1493.0       699.0   

   uninsured_adults  with_disability  no_vehicle_households  
0             516.0          16594.0                  262.0  
1            1000.0          72481.0  

We need to merge in zip code but before we do that we need to group the healthcare_df by zip code and get a facility count

In [45]:
# Force both to strings and pad with zeros to ensure '07701' doesn't become '7701'
census_df['zip_code'] = census_df['zip_code'].astype(str).str.strip().str.zfill(5)
healthcare_df['zip_cd'] = healthcare_df['zip_cd'].astype(str).str.strip().str.zfill(5)

# Group healthcare facilities by ZIP to get a count
healthcare_counts = healthcare_df.groupby('zip_cd').size().reset_index(name='facility_count')

# Double check the sample now
print(f"Sample Census ZIP: '{census_df['zip_code'].iloc[0]}'")
print(f"Sample Healthcare ZIP: '{healthcare_counts['zip_cd'].iloc[0]}'")

Sample Census ZIP: '75001'
Sample Healthcare ZIP: '00601'


In [46]:
# Merge census data with healthcare facility counts on ZIP code
merged_df = pd.merge(
    census_df, 
    healthcare_counts, 
    left_on='zip_code', 
    right_on='zip_cd', 
    how='left'
)

# fill in missing facility counts with 0 (indicating no facilities in that ZIP code)
merged_df['facility_count'] = merged_df['facility_count'].fillna(0)

# Drop the redundant column from the merge
if 'zip_cd' in merged_df.columns:
    merged_df = merged_df.drop(columns=['zip_cd'])

print("Merge Complete!")
print(merged_df.isnull().sum())

Merge Complete!
zip_code                     0
median_age                  82
total_population            56
median_household_income    227
bachelors_degree           170
masters_degree             271
below_poverty              203
unemployed                 364
uninsured_adults           395
with_disability             69
no_vehicle_households      400
facility_count               0
dtype: int64


In Census data, `NaN` values usually occur in ZIP codes with very small populations, industrial areas (like refineries in East Houston), or parks where people don't live.
The first thing we will address in droping the Ghost zip Codes by removing rows where total_population is 0 or where core features (median_age, median_household_income) are missing.

In [47]:
# Drop rows where critical modeling variables are missing
# These ZIPs aren't useful for a demand-supply model
df_clean = merged_df.dropna(subset=['median_age', 'total_population', 'median_household_income'])

print(f"Rows remaining after dropping critical NAs: {len(df_clean)}")

Rows remaining after dropping critical NAs: 1762


For features like `no_vehicle_households` or `unemployed`, where we have ~400 missing values we can fill the NaN with the Median of the rest of Houston. Income and employment data are often skewed; the median is more robust than the mean.

In [48]:
# Fill secondary features with the median
secondary_cols = ['unemployed', 'uninsured_adults', 'no_vehicle_households', 'with_disability']
for col in secondary_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())
print(f"Rows remaining after filling secondary NAs: {len(df_clean)}")

Rows remaining after filling secondary NAs: 1762


C:\Users\willi\AppData\Local\Temp\ipykernel_28168\3210357992.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean[col] = df_clean[col].fillna(df_clean[col].median())
C:\Users\willi\AppData\Local\Temp\ipykernel_28168\3210357992.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean[col] = df_clean[col].fillna(df_clean[col].median())
C:\Users\willi\AppData\Local\Temp\ipykernel_28168\3210357992.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try u

We must also create a facility_density variable. We need to handle the case where population is 0 to avoid inf values.

In [49]:
# Only calculate for ZIPs with population > 0
df_clean = df_clean[df_clean['total_population'] > 0].copy()

# Calculate Density: Facilities per 1,000 residents
df_clean['facility_density'] = (df_clean['facility_count'] / df_clean['total_population']) * 1000

# Cap extreme outliers (optional but good for regression)
# Some industrial ZIPs with 10 people and 1 clinic create huge density numbers
q_99 = df_clean['facility_density'].quantile(0.99)
df_clean['facility_density'] = df_clean['facility_density'].clip(upper=q_99)